In [1]:
import os
print(os.environ.get('TRAJECTORY_WORKDIR'))
print(os.environ.get('PROJECT_TAG'))


/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da
None


In [2]:
import os
for root, dirs, files in os.walk('/workspace'):
    # skip memory dirs and trajectory subdirs
    depth = root.count('/') - 1
    if depth > 3: continue
    for f in files:
        p = os.path.join(root, f)
        try:
            sz = os.path.getsize(p)
        except: sz = -1
        if sz < 5_000_000:
            print(p, sz)


/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/test_module.kicad_sym 21950
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/.kernel_llm_logs_1.txt 0
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/test_module.kicad_pro 941
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/test_module.kicad_sch 53537
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/sym-lib-table 157
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/generate.py 30538
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/README.md 16584
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/.prompts/HEARTBEAT.md 1997
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/.prompts/MEMORY.md 379
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/.prompts/USER.md 617
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/.prompts/BOOTSTRAP.md 3942
/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/memory/4d8e2444-cf03-4d71-871d-2ade3b9188da_memory_heron_c0_20260608T224707404491.jsonl 38316
/workspace/.kernel_meta_4d8e2444-cf03-4d71-871d-2ade3b9188da/kernel-1.jso

In [3]:
import re, collections
sch = open('/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/test_module.kicad_sch').read()
# Count components, footprints, nets
refs = re.findall(r'\(reference "([^"]+)"\)', sch)
nets = set(re.findall(r'\(global_label "([^"]+)"', sch))
footprints = re.findall(r'\(property "Footprint" "([^"]*)"', sch)
print("References:", sorted(set(refs)))
print("Number of unique components:", len(set(refs)))
print("Nets:", sorted(nets))
print("Distinct footprint strings:", set(footprints))
print("Has kicad_pcb file:", any(f.endswith('.kicad_pcb') for f in __import__('os').listdir('/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da')))


References: ['C1', 'C2', 'C3', 'J1', 'M1', 'M2', 'M3', 'SOL1', 'SR1', 'U1', 'U2', 'U3', 'U4', 'U5']
Number of unique components: 14
Nets: ['+12V', '+3V3', '+5V', 'GND', 'HAPT_EN', 'I2C_SCL', 'I2C_SDA', 'SERVO_SIG', 'SOL_A', 'SOL_B', 'SOL_IN1', 'SOL_IN2', 'STP_A1', 'STP_A2', 'STP_B1', 'STP_B2', 'STP_RX', 'STP_TX', 'VIB_A', 'VIB_B']
Distinct footprint strings: {''}
Has kicad_pcb file: False


In [4]:
# Confirm: zero footprints assigned anywhere
print(sch.count('(property "Footprint" "")'))  # all empty
print(sch.count('.kicad_pcb'))
# load_cell / HX711 mentions
print('HX711 in sch?', 'HX711' in sch, 'in readme?', 'HX711' in open('/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/README.md').read())


0
0
HX711 in sch? False in readme? False


In [5]:
# So even properties hidden -- let me check raw
print('(property "Footprint"' in sch, sch.count('Footprint'))
# Find one
i = sch.find('Footprint')
print(sch[i-20:i+200])


True 26
1.27))))
(property "Footprint" "" (id 2) (at 0 0 0)
  (effects (font (size 1.27 1.27)) hide))
(property "Datasheet" "" (id 3) (at 0 0 0)
  (effects (font (size 1.27 1.27)) hide))
          (symbol "Pi_Pico_W_0_1"
       


In [6]:
# Confirm all footprint properties are empty strings
hits = re.findall(r'\(property "Footprint" "([^"]*)"', sch)
print("Total footprint properties:", len(hits), "non-empty:", sum(1 for h in hits if h))


Total footprint properties: 26 non-empty: 0


In [7]:
# Check kicad_pro for board / pcbnew settings
pro = open('/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/test_module.kicad_pro').read()
print(pro)


{
  "board": {},
  "boards": [],
  "cvpcb": {"equivalence_files": []},
  "erc": {"erc_exclusions": [], "meta": {"version": 0}},
  "libraries": {"pinned_footprint_libs": [], "pinned_symbol_libs": []},
  "meta": {"filename": "test_module.kicad_pro", "version": 1},
  "net_settings": {"classes": [], "meta": {"version": 3}, "net_colors": null},
  "pcbnew": {"page_layout_descr_file": ""},
  "schematic": {
    "annotate_start_num": 0,
    "drawing": {"default_line_thickness": 6.0, "default_text_size": 50.0},
    "legacy_lib_dir": "",
    "legacy_lib_list": [],
    "meta": {"version": 1},
    "net_format_name": "Pcbnew",
    "ngspice": {"meta": {"version": 0}, "workbook_filename": ""},
    "page_layout_descr_file": "",
    "plot_directory": "",
    "spice_external_command": "spice \"%I\"",
    "subpart_first_id": 65,
    "subpart_id_separator": 0
  },
  "sheets": [["00000000-0000-0000-0003-000000000001", ""]],
  "text_variables": {}
}



In [8]:
# Look at sym-lib-table
print(open('/workspace/4d8e2444-cf03-4d71-871d-2ade3b9188da/sym-lib-table').read())


(sym_lib_table
  (lib (name "test_module")(type "KiCad")(uri "${KIPRJMOD}/test_module.kicad_sym")(options "")(descr "Powder-doser test-module breakouts"))
)



In [9]:
# Save a structured notes file for the workspace before submitting
import os, json, textwrap
workdir = os.environ['TRAJECTORY_WORKDIR']
notes = {
    "components_in_schematic": ['C1','C2','C3','J1','M1','M2','M3','SOL1','SR1','U1','U2','U3','U4','U5'],
    "n_unique_components": 14,
    "nets": sorted(['+12V','+3V3','+5V','GND','HAPT_EN','I2C_SCL','I2C_SDA','SERVO_SIG','SOL_A','SOL_B','SOL_IN1','SOL_IN2','STP_A1','STP_A2','STP_B1','STP_B2','STP_RX','STP_TX','VIB_A','VIB_B']),
    "n_nets": 20,
    "footprint_properties_populated": 0,
    "footprint_properties_total": 26,
    "has_kicad_pcb": False,
    "has_board_outline": False,
    "has_fp_lib_table": False,
}
with open(f"{workdir}/audit_notes.json","w") as f:
    json.dump(notes, f, indent=2)
print(json.dumps(notes, indent=2))


{
  "components_in_schematic": [
    "C1",
    "C2",
    "C3",
    "J1",
    "M1",
    "M2",
    "M3",
    "SOL1",
    "SR1",
    "U1",
    "U2",
    "U3",
    "U4",
    "U5"
  ],
  "n_unique_components": 14,
  "nets": [
    "+12V",
    "+3V3",
    "+5V",
    "GND",
    "HAPT_EN",
    "I2C_SCL",
    "I2C_SDA",
    "SERVO_SIG",
    "SOL_A",
    "SOL_B",
    "SOL_IN1",
    "SOL_IN2",
    "STP_A1",
    "STP_A2",
    "STP_B1",
    "STP_B2",
    "STP_RX",
    "STP_TX",
    "VIB_A",
    "VIB_B"
  ],
  "n_nets": 20,
  "footprint_properties_populated": 0,
  "footprint_properties_total": 26,
  "has_kicad_pcb": false,
  "has_board_outline": false,
  "has_fp_lib_table": false
}
